In [ ]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
def char_len_bucket(n):
    if n <= 20:
        return "000-020"
    if n <= 40:
        return "021-040"
    if n <= 60:
        return "041-060"
    if n <= 80:
        return "061-080"
    if n <= 120:
        return "081-120"
    return "121+"

def token_len_bucket(n):
    if n <= 5:
        return "01-05"
    if n <= 10:
        return "06-10"
    if n <= 15:
        return "11-15"
    if n <= 20:
        return "16-20"
    if n <= 30:
        return "21-30"
    return "31+"

def asymmetry_bucket(diff):
    if diff <= 2:
        return "00-02"
    if diff <= 5:
        return "03-05"
    if diff <= 10:
        return "06-10"
    if diff <= 20:
        return "11-20"
    return "21+"

df["sentence1_char_len"] = df["sentence1"].str.len().astype(np.int32)
df["sentence2_char_len"] = df["sentence2"].str.len().astype(np.int32)
df["sentence1_token_len"] = df["sentence1"].str.split().str.len().astype(np.int32)
df["sentence2_token_len"] = df["sentence2"].str.split().str.len().astype(np.int32)

df["sentence1_char_bucket"] = df["sentence1_char_len"].map(char_len_bucket)
df["sentence2_char_bucket"] = df["sentence2_char_len"].map(char_len_bucket)
df["sentence1_token_bucket"] = df["sentence1_token_len"].map(token_len_bucket)
df["sentence2_token_bucket"] = df["sentence2_token_len"].map(token_len_bucket)

df["pair_mean_char_len"] = ((df["sentence1_char_len"] + df["sentence2_char_len"]) / 2.0).astype(np.float32)
df["pair_mean_token_len"] = ((df["sentence1_token_len"] + df["sentence2_token_len"]) / 2.0).astype(np.float32)
df["pair_abs_char_len_diff"] = (df["sentence1_char_len"] - df["sentence2_char_len"]).abs().astype(np.int32)
df["pair_abs_token_len_diff"] = (df["sentence1_token_len"] - df["sentence2_token_len"]).abs().astype(np.int32)
df["pair_char_bucket"] = df["pair_mean_char_len"].round().astype(int).map(char_len_bucket)
df["pair_token_bucket"] = df["pair_mean_token_len"].round().astype(int).map(token_len_bucket)
df["pair_len_asymmetry_bucket"] = df["pair_abs_token_len_diff"].map(asymmetry_bucket)

print(df[[
    "sentence1_char_len", "sentence2_char_len",
    "sentence1_token_len", "sentence2_token_len",
    "pair_char_bucket", "pair_token_bucket", "pair_len_asymmetry_bucket"
]].head())

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
sentence_counts = Counter(all_sentences)
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

unique_df = pd.DataFrame({"sentence": unique_sentences})
unique_df["sentence_idx"] = np.arange(len(unique_df), dtype=np.int32)
unique_df["char_len"] = unique_df["sentence"].str.len().astype(np.int32)
unique_df["token_len"] = unique_df["sentence"].str.split().str.len().astype(np.int32)
unique_df["char_bucket"] = unique_df["char_len"].map(char_len_bucket)
unique_df["token_bucket"] = unique_df["token_len"].map(token_len_bucket)
unique_df["occurrences"] = unique_df["sentence"].map(sentence_counts).astype(np.int32)

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
})
print(unique_df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()

token_bucket_order = ["01-05", "06-10", "11-15", "16-20", "21-30", "31+"]
char_bucket_order = ["000-020", "021-040", "041-060", "061-080", "081-120", "121+"]

unique_df["token_bucket"] = pd.Categorical(unique_df["token_bucket"], categories=token_bucket_order, ordered=True)
unique_df["char_bucket"] = pd.Categorical(unique_df["char_bucket"], categories=char_bucket_order, ordered=True)
unique_df_sorted = unique_df.sort_values(["token_bucket", "char_bucket", "token_len", "char_len", "sentence_idx"]).reset_index(drop=True)

sorted_sentences = unique_df_sorted["sentence"].tolist()
sorted_embeddings = model.encode(
    sorted_sentences,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

embedding_dim = int(sorted_embeddings.shape[1])
unique_embeddings = np.empty((num_unique_sentences, embedding_dim), dtype=np.float32)
unique_embeddings[unique_df_sorted["sentence_idx"].to_numpy()] = sorted_embeddings.astype(np.float32)

print({
    "model_name": model_name,
    "embedding_shape": tuple(unique_embeddings.shape),
    "num_unique_sentences": num_unique_sentences,
})

In [ ]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["cosine_similarity"] = cosine_similarity.astype(np.float32)
results_df["predicted_score_0_5"] = predicted_score_0_5.astype(np.float32)
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"]).astype(np.float32)
results_df["squared_error"] = ((results_df["predicted_score_0_5"] - results_df["label"]) ** 2).astype(np.float32)

print(results_df[[
    "sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5",
    "pair_char_bucket", "pair_token_bucket", "pair_len_asymmetry_bucket"
]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], labels).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], labels).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

def safe_pearson(x, y):
    if len(x) < 2:
        return np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(pearsonr(x, y).statistic)

def safe_spearman(x, y):
    if len(x) < 2:
        return np.nan
    if pd.Series(x).nunique() < 2 or pd.Series(y).nunique() < 2:
        return np.nan
    return float(spearmanr(x, y).statistic)

def bucket_metrics(frame, bucket_col):
    rows = []
    for bucket_value, group in frame.groupby(bucket_col, dropna=False):
        y_true = group["label"].to_numpy(dtype=np.float32)
        y_pred = group["predicted_score_0_5"].to_numpy(dtype=np.float32)
        rows.append({
            "bucket": str(bucket_value),
            "num_examples": int(len(group)),
            "label_mean": float(np.mean(y_true)),
            "pred_mean": float(np.mean(y_pred)),
            "mae": float(np.mean(np.abs(y_pred - y_true))),
            "rmse": float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
            "pearson": safe_pearson(y_pred, y_true),
            "spearman": safe_spearman(y_pred, y_true),
        })
    return pd.DataFrame(rows).sort_values(["num_examples", "bucket"], ascending=[False, True]).reset_index(drop=True)

pair_char_bucket_metrics = bucket_metrics(results_df, "pair_char_bucket")
pair_token_bucket_metrics = bucket_metrics(results_df, "pair_token_bucket")
pair_asymmetry_bucket_metrics = bucket_metrics(results_df, "pair_len_asymmetry_bucket")

long_df = pd.concat([
    results_df[["sentence1", "predicted_score_0_5"]].rename(columns={"sentence1": "sentence"}),
    results_df[["sentence2", "predicted_score_0_5"]].rename(columns={"sentence2": "sentence"}),
], ignore_index=True)

repeated_sentence_stats = (
    long_df.groupby("sentence")
    .agg(
        occurrences=("sentence", "size"),
        min_predicted_score=("predicted_score_0_5", "min"),
        max_predicted_score=("predicted_score_0_5", "max"),
        mean_predicted_score=("predicted_score_0_5", "mean"),
    )
    .reset_index()
)
repeated_sentence_stats = repeated_sentence_stats[repeated_sentence_stats["occurrences"] > 1].copy()
repeated_sentence_stats["score_range"] = repeated_sentence_stats["max_predicted_score"] - repeated_sentence_stats["min_predicted_score"]
repeated_sentence_stats = repeated_sentence_stats.sort_values(
    by=["occurrences", "score_range", "sentence"], ascending=[False, False, True]
).reset_index(drop=True)

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae": round(mae, 6),
    "rmse": round(rmse, 6),
})
print(pair_char_bucket_metrics)
print(pair_token_bucket_metrics)
print(pair_asymmetry_bucket_metrics)
print(repeated_sentence_stats.head(10))

In [ ]:
runtime_seconds = time.time() - start_time

unique_bucket_summary = (
    unique_df.groupby(["char_bucket", "token_bucket"], observed=False)
    .agg(
        num_unique_sentences=("sentence", "size"),
        total_occurrences=("occurrences", "sum"),
        avg_occurrences=("occurrences", "mean"),
    )
    .reset_index()
    .sort_values(["num_unique_sentences", "char_bucket", "token_bucket"], ascending=[False, True, True])
    .reset_index(drop=True)
)

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"pearson_correlation: {float(pearson_corr):.6f}")
print(f"spearman_correlation: {float(spearman_corr):.6f}")
print(f"mae: {mae:.6f}")
print(f"rmse: {rmse:.6f}")
print(f"num_repeated_sentences: {len(repeated_sentence_stats)}")
print("top_unique_sentence_buckets:")
print(unique_bucket_summary.head(10).to_dict(orient="records"))
print("pair_char_bucket_metrics:")
print(pair_char_bucket_metrics.to_dict(orient="records"))
print("pair_token_bucket_metrics:")
print(pair_token_bucket_metrics.to_dict(orient="records"))
print("pair_len_asymmetry_bucket_metrics:")
print(pair_asymmetry_bucket_metrics.to_dict(orient="records"))
print("most_repeated_sentences_with_score_ranges:")
print(repeated_sentence_stats.head(10).to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")